# 08. Double ML

## Background

Double Machine Learning (Chernozhukov et al. 2018) is the state-of-the-art method for estimating treatment effects when you have many covariates and want to use flexible ML models for nuisance functions. It generalizes AIPW to the continuous treatment case and adds a crucial twist: **cross-fitting** (sample splitting) to avoid regularization bias.

The key insight: when you use the same data to fit your nuisance models (propensity score, outcome model) and estimate your treatment effect, regularization bias from the ML models leaks into your estimates. Cross-fitting — fitting on held-out folds — eliminates this bias.

Double ML achieves the parametric rate n^{-1/2} for the treatment effect even when the nuisance models converge at slower rates, as long as their product of errors is o(n^{-1/2}) — the Neyman orthogonality condition.

## What You'll Learn

- The Partially Linear Model: Y = τD + g(X) + ε, E[ε|D,X] = 0
- Partialling out: residualize Y and D on X, then regress residuals
- Neyman orthogonality: why it eliminates first-order bias from nuisance errors
- Cross-fitting: K-fold sample splitting to avoid Donsker conditions
- The `DoubleML` package: production implementation with inference
- Continuous vs binary treatment; ATT vs ATE in Double ML

## Why This Matters

Double ML is now standard in tech industry causal inference (Uber, Airbnb, Microsoft). The `DoubleML` Python package (Bach et al. 2022) implements the full suite with automatic cross-fitting, clustered SEs, and sensitivity analysis. It also supports the Interactive Regression Model for heterogeneous effects — a precursor to causal forests.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LassoCV, LogisticRegressionCV, Ridge
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. The Partially Linear Model

Y_i = τ · D_i + g(X_i) + ε_i

where D is the treatment (can be continuous or binary), g(X) is an unknown function of high-dimensional controls, and τ is the scalar we want.

**Key step — partialling out (Frisch-Waugh-Lovell generalized)**:

Ẽ_i = Y_i − m̂(X_i)   (residual from predicting Y with X)
Ṽ_i = D_i − l̂(X_i)   (residual from predicting D with X)

Then τ̂ = (Σ Ṽ_i·Ẽ_i) / (Σ Ṽ_i²)

In [ ]:
def simulate_plm(n=2000, p=20, tau=3.0, seed=0):
    """
    Partially Linear Model DGP.
    p covariates, non-linear g(X) and l(X), true effect = tau.
    """
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, (n, p))
    # Non-linear nuisance functions
    g_X = np.sin(X[:, 0]) + X[:, 1]**2 + 0.5*X[:, 2] + X[:, 3]*X[:, 4]
    l_X = 0.3*X[:, 0] + np.tanh(X[:, 1]) + 0.2*X[:, 2]**2  # E[D|X]

    D = l_X + rng.normal(0, 1, n)   # continuous treatment
    Y = tau * D + g_X + rng.normal(0, 1, n)
    return X, D, Y, g_X, l_X


X, D, Y, g_X, l_X = simulate_plm(n=3000, p=20, tau=3.0)
print(f"True ATE (τ) = 3.0")
print(f"n={len(Y)}, p={X.shape[1]}")

# Naive OLS (biased if g(X) is misspecified)
Xd = np.column_stack([D, X])
from numpy.linalg import lstsq
beta_ols = lstsq(np.column_stack([np.ones(len(Y)), Xd]), Y, rcond=None)[0]
print(f"Linear OLS estimate = {beta_ols[1]:.3f}  (works here but not in general)")

## 2. Double ML with Cross-Fitting

The key steps:
1. Split data into K folds
2. For each fold k: train nuisance models on the other K-1 folds
3. Generate out-of-fold predictions for fold k
4. Residualize: Ẽ = Y − m̂(X), Ṽ = D − l̂(X)
5. OLS Ẽ ~ Ṽ to get τ̂

In [ ]:
def double_ml(Y, D, X, n_folds=5, model_y=None, model_d=None, seed=0):
    """
    Double ML via K-fold cross-fitting.
    model_y: sklearn estimator for E[Y|X] (outcome nuisance)
    model_d: sklearn estimator for E[D|X] (treatment nuisance)
    Returns: tau_hat, se, (Y_res, D_res)
    """
    rng = np.random.default_rng(seed)
    n = len(Y)

    if model_y is None:
        model_y = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)
    if model_d is None:
        model_d = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)

    Y_hat = np.zeros(n)
    D_hat = np.zeros(n)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for train_idx, val_idx in kf.split(X):
        # Fit on train, predict on val (cross-fitting)
        my = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)
        md_ = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)

        my.fit(X[train_idx], Y[train_idx])
        md_.fit(X[train_idx], D[train_idx])

        Y_hat[val_idx] = my.predict(X[val_idx])
        D_hat[val_idx] = md_.predict(X[val_idx])

    # Residuals
    Y_res = Y - Y_hat
    D_res = D - D_hat

    # OLS of Y_res on D_res
    tau_hat = np.dot(D_res, Y_res) / np.dot(D_res, D_res)

    # Sandwich SE
    psi = D_res * (Y_res - tau_hat * D_res)
    se = np.sqrt(np.mean(psi**2) / (np.mean(D_res**2)**2 * n))

    return tau_hat, se, Y_res, D_res


tau_dml, se_dml, Y_res, D_res = double_ml(Y, D, X, n_folds=5)
ci_lo, ci_hi = tau_dml - 1.96*se_dml, tau_dml + 1.96*se_dml
print(f"Double ML estimate = {tau_dml:.3f}  (SE={se_dml:.3f})")
print(f"95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"True τ = 3.000")

In [ ]:
# Visualize the partialling-out step
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw Y vs D — obscured by g(X)
axes[0].scatter(D[:300], Y[:300], alpha=0.3, s=10, color='steelblue')
axes[0].set_xlabel('D (treatment)'); axes[0].set_ylabel('Y (outcome)')
axes[0].set_title(f'Raw: hard to see τ (OLS slope ≈ {np.polyfit(D,Y,1)[0]:.3f})')

# Residuals — clean signal
axes[1].scatter(D_res[:300], Y_res[:300], alpha=0.3, s=10, color='#F44336')
m, b = np.polyfit(D_res, Y_res, 1)
x_range = np.linspace(D_res.min(), D_res.max(), 100)
axes[1].plot(x_range, m*x_range + b, 'k-', lw=2, label=f'Slope = {m:.3f} (≈ τ)')
axes[1].set_xlabel('D̃ (D residualized on X)'); axes[1].set_ylabel('Ỹ (Y residualized on X)')
axes[1].set_title('After partialling out X — clean causal signal')
axes[1].legend()

plt.tight_layout()
plt.savefig('08_partialling_out.png', dpi=110, bbox_inches='tight')
plt.show()

## 3. Neyman Orthogonality — Why Cross-Fitting Matters

Without cross-fitting, regularization bias in the nuisance models propagates to τ̂. With cross-fitting, the influence of nuisance errors on τ̂ is second-order: proportional to (ε_Y · ε_D) where both are estimation errors — a product that vanishes faster.

In [ ]:
def single_sample_ml(Y, D, X, seed=0):
    """Double ML WITHOUT cross-fitting (biased due to overfitting)."""
    my = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed)
    md = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed)
    my.fit(X, Y); md.fit(X, D)
    Y_res = Y - my.predict(X)
    D_res = D - md.predict(X)
    tau_hat = np.dot(D_res, Y_res) / np.dot(D_res, D_res)
    return tau_hat


n_sims = 100
tau_dml_sims, tau_no_cf_sims = [], []
for sim in range(n_sims):
    X_s, D_s, Y_s, _, _ = simulate_plm(n=500, p=20, tau=3.0, seed=sim)
    tau_dml_sims.append(double_ml(Y_s, D_s, X_s, n_folds=5, seed=sim)[0])
    tau_no_cf_sims.append(single_sample_ml(Y_s, D_s, X_s, seed=sim))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tau_dml_sims, bins=25, alpha=0.7, color='#4CAF50', label=f'Double ML (cross-fit): mean={np.mean(tau_dml_sims):.3f}')
ax.hist(tau_no_cf_sims, bins=25, alpha=0.7, color='#F44336', label=f'No cross-fitting: mean={np.mean(tau_no_cf_sims):.3f}')
ax.axvline(3.0, color='black', lw=2.5, label='True τ=3.0')
ax.set_xlabel('τ̂ estimate'); ax.set_ylabel('Count')
ax.set_title('Cross-fitting eliminates regularization bias (100 simulations, n=500)')
ax.legend()
plt.tight_layout()
plt.savefig('08_crossfitting.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"With cross-fitting:    bias = {np.mean(tau_dml_sims) - 3:.3f}, rmse = {np.sqrt(np.mean((np.array(tau_dml_sims)-3)**2)):.3f}")
print(f"Without cross-fitting: bias = {np.mean(tau_no_cf_sims) - 3:.3f}, rmse = {np.sqrt(np.mean((np.array(tau_no_cf_sims)-3)**2)):.3f}")

## 4. DoubleML Package

The `doubleml` package provides a production-ready implementation with automatic cross-fitting, multiple ML learners, and inference via the score function approach.

In [ ]:
# DoubleML package walkthrough (install: pip install doubleml)
try:
    import doubleml as dml
    from doubleml import DoubleMLData, DoubleMLPLR

    X_s, D_s, Y_s, _, _ = simulate_plm(n=2000, p=20, tau=3.0, seed=0)
    df_dml = pd.DataFrame(X_s, columns=[f'x{i}' for i in range(X_s.shape[1])])
    df_dml['D'] = D_s
    df_dml['Y'] = Y_s

    data = DoubleMLData(df_dml, y_col='Y', d_cols='D')

    ml_y = GradientBoostingRegressor(n_estimators=100, max_depth=3)
    ml_d = GradientBoostingRegressor(n_estimators=100, max_depth=3)

    plr = DoubleMLPLR(data, ml_l=ml_y, ml_m=ml_d, n_folds=5)
    plr.fit()
    print("=== DoubleML PLR Results ===")
    print(plr.summary)

except ImportError:
    print("doubleml not installed — showing manual implementation results instead")
    print(f"Manual Double ML: τ̂ = {tau_dml:.3f}  SE={se_dml:.3f}")
    print(f"True τ = 3.000")
    print("\nInstall with: pip install doubleml")
    print("DoubleML also supports:")
    print("  - DoubleMLIRM (interactive regression model, binary treatment)")
    print("  - DoubleMLIIVM (IV version with instrument)")
    print("  - Clustered SEs, heterogeneous effects via CATE")